# Actividad 2 - Procesamiento de datos en infraestructura cloud

## Estudiante
Jorge Paniagua

## Plataforma
Databricks Community Edition (Free Edition)

## Objetivo
Desarrollar un proceso de ingestión, almacenamiento y validación de datos utilizando Spark y SQL en Databricks, implementando un esquema estructurado y realizando análisis comparativos entre SQL y PySpark.

# Contenido

1. Diseño del esquema de datos
2. Configuración de infraestructura en Databricks
3. Obtención e ingestión de datos desde Kaggle
4. Validaciones con Spark y SQL
5. Comparación entre SQL y Spark
6. Conclusiones


In [0]:
# Ruta del archivo CSV
ruta_csv = "/Workspace/Users/mario.paniagua@est.iudigital.edu.co/Actividad_2/netflix_titles.csv"

# Lectura del dataset
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(ruta_csv)

# Mostrar registros
df.show(5)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         NULL|Septem

# 1. Obtención e ingestión de datos desde Kaggle

Para esta actividad se utilizó el dataset **Netflix Movies and TV Shows** obtenido desde Kaggle.

El archivo fue cargado manualmente al entorno de Databricks y posteriormente leído utilizando Spark DataFrame API.

La lectura se realizó en formato CSV utilizando encabezados e inferencia automática de esquema.

# 2. Diseño del esquema de datos

El dataset seleccionado contiene información sobre películas y series disponibles en Netflix.

Las entidades principales corresponden a contenidos audiovisuales, incluyendo información relacionada con:

- Tipo de contenido
- Título
- Director
- Actores
- País
- Año de lanzamiento
- Clasificación
- Duración
- Categorías

La columna `show_id` se utilizará como identificador único del registro.

## Diccionario de datos

| Campo | Tipo de dato | Nulo permitido | Descripción |
|---|---|---|---|
| show_id | STRING | No | Identificador único del contenido |
| type | STRING | No | Tipo de contenido (Movie o TV Show) |
| title | STRING | No | Nombre del contenido |
| director | STRING | Sí | Director del contenido |
| cast | STRING | Sí | Actores principales |
| country | STRING | Sí | País de origen |
| date_added | STRING | Sí | Fecha de ingreso a Netflix |
| release_year | INT | No | Año de lanzamiento |
| rating | STRING | Sí | Clasificación por edad |
| duration | STRING | Sí | Duración o temporadas |
| listed_in | STRING | Sí | Géneros o categorías |
| description | STRING | Sí | Descripción del contenido |

## Diagrama del esquema

```mermaid
erDiagram
    NETFLIX_TITLES {
        string show_id PK
        string type
        string title
        string director
        string cast
        string country
        string date_added
        int release_year
        string rating
        string duration
        string listed_in
        string description
    }
```

## Definición del esquema usando Spark SQL (DDL)

```sql
CREATE TABLE netflix_titles (
    show_id STRING NOT NULL,
    type STRING NOT NULL,
    title STRING NOT NULL,
    director STRING,
    cast STRING,
    country STRING,
    date_added STRING,
    release_year INT NOT NULL,
    rating STRING,
    duration STRING,
    listed_in STRING,
    description STRING
);
```

# 3. Configuración de infraestructura en Databricks

Para el desarrollo de la actividad se utilizó Databricks Community Edition (Free Edition) con un entorno Serverless administrado automáticamente por la plataforma.

Se utilizó Apache Spark como motor de procesamiento distribuido y Python como lenguaje principal para el análisis y transformación de datos.

In [0]:
# Mostrar versión de Spark
print("Versión de Spark:")
print(spark.version)

Versión de Spark:
4.1.0


In [0]:
# Mostrar configuraciones disponibles de Spark
configs = spark.conf.getAll

for key, value in configs.items():
    print(f"{key} = {value}")

spark.databricks.execution.timeout = 9000
spark.sql.ansi.enabled = true
spark.sql.shuffle.partitions = auto


In [0]:
import sys

print("Versión de Python:")
print(sys.version)

Versión de Python:
3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]


In [0]:
# Mostrar directorios disponibles en DBFS
display(dbutils.fs.ls("/"))

path,name,size,modificationTime
dbfs:/Volumes/,Volumes/,0,0
dbfs:/Workspace/,Workspace/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0


In [0]:
# Crear tabla persistente
df.write.mode("overwrite").saveAsTable("netflix_titles")

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,netflix_titles,false


In [0]:
%sql
DESCRIBE TABLE netflix_titles;

col_name,data_type,comment
show_id,string,null
type,string,null
title,string,null
director,string,null
cast,string,null
country,string,null
date_added,string,null
release_year,string,null
rating,string,null
duration,string,null


In [0]:
%sql
SHOW CREATE TABLE netflix_titles;

createtab_stmt
"CREATE TABLE workspace.default.netflix_titles ( show_id STRING COLLATE UTF8_BINARY, type STRING COLLATE UTF8_BINARY, title STRING COLLATE UTF8_BINARY, director STRING COLLATE UTF8_BINARY, cast STRING COLLATE UTF8_BINARY, country STRING COLLATE UTF8_BINARY, date_added STRING COLLATE UTF8_BINARY, release_year STRING COLLATE UTF8_BINARY, rating STRING COLLATE UTF8_BINARY, duration STRING COLLATE UTF8_BINARY, listed_in STRING COLLATE UTF8_BINARY, description STRING COLLATE UTF8_BINARY) USING delta TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


In [0]:
# Mostrar esquema del DataFrame
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [0]:
# Estadísticas descriptivas
df.describe().show()

+-------+--------------------+-------------+---------------------------------+--------------------+--------------------+----------------+---------------+-----------------+-----------------+-------------+--------------------+--------------------+
|summary|             show_id|         type|                            title|            director|                cast|         country|     date_added|     release_year|           rating|     duration|           listed_in|         description|
+-------+--------------------+-------------+---------------------------------+--------------------+--------------------+----------------+---------------+-----------------+-----------------+-------------+--------------------+--------------------+
|  count|                8809|         8808|                             8807|                6173|                7983|            7977|           8796|             8807|             8803|         8804|                8806|                8806|
|   mean|       

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM netflix_titles;

total_registros
8809


In [0]:
%sql
SELECT *
FROM netflix_titles
LIMIT 10;

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,null,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
s2,TV Show,Blood & Water,null,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",null,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."
s4,TV Show,Jailbirds New Orleans,null,null,null,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice Center in New Orleans on this gritty reality series."
s5,TV Show,Kota Factory,null,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaas Channa, Revathi Pillai, Urvi Singh, Arun Kumar",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV Comedies","In a city of coaching centers known to train India’s finest collegiate minds, an earnest but unexceptional student and his friends navigate campus life."
s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, Henry Thomas, Kristin Lehman, Samantha Sloyan, Igby Rigney, Rahul Kohli, Annarah Cymone, Annabeth Gish, Alex Essoe, Rahul Abburi, Matt Biedel, Michael Trucco, Crystal Balint, Louis Oliver",null,"September 24, 2021",2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries","The arrival of a charismatic young priest brings glorious miracles, ominous mysteries and renewed religious fervor to a dying town desperate to believe."
s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, Sofia Carson, Liza Koshy, Ken Jeong, Elizabeth Perkins, Jane Krakowski, Michael McKean, Phil LaMarr",null,"September 24, 2021",2021,PG,91 min,Children & Family Movies,"Equestria's divided. But a bright-eyed hero believes Earth Ponies, Pegasi and Unicorns should be pals — and, hoof to heart, she’s determined to prove it."
s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra Duah, Nick Medley, Mutabaruka, Afemo Omilami, Reggie Carter, Mzuri","United States, Ghana, Burkina Faso, United Kingdom, Germany, Ethiopia","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model slips back in time, becomes enslaved on a plantation and bears witness to the agony of her ancestral past."
s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Hollywood",United Kingdom,"September 24, 2021",2021,TV-14,9 Seasons,"British TV Shows, Reality TV","A talented batch of amateur bakers face off in a 10-week competition, whipping up their best dishes in the hopes of being named the U.K.'s best."
s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, Timothy Olyphant, Daveed Diggs, Skyler Gisondo, Laura Harrier, Rosalind Chao, 

In [0]:
%sql
SELECT title, country, release_year, rating
FROM netflix_titles
WHERE country = 'United States'
LIMIT 10;

title,country,release_year,rating
Dick Johnson Is Dead,United States,2020,PG-13
The Starling,United States,2021,PG-13
Dear White People,United States,2021,TV-MA
Grown Ups,United States,2010,PG-13
Dark Skies,United States,2013,PG-13
He-Man and the Masters of the Universe,United States,2021,TV-Y7
Jaws,United States,1975,PG
Jaws 2,United States,1978,PG
Jaws 3,United States,1983,PG
Jaws: The Revenge,United States,1987,PG-13


In [0]:
%sql
SELECT type, COUNT(*) AS cantidad
FROM netflix_titles
GROUP BY type;

type,cantidad
Movie,6131
TV Show,2676
null,1
William Wyler,1


In [0]:
# Agrupación usando PySpark
df.groupBy("type").count().show()

+-------------+-----+
|         type|count|
+-------------+-----+
|      TV Show| 2676|
|        Movie| 6131|
|         NULL|    1|
|William Wyler|    1|
+-------------+-----+



## Interpretación de resultados

Las validaciones realizadas en Spark y SQL permitieron confirmar:

- La correcta carga del dataset en el entorno Databricks.
- La creación exitosa de la tabla persistente `netflix_titles`.
- La correcta detección de columnas y tipos de datos.
- La posibilidad de realizar consultas analíticas utilizando SQL y PySpark.

Durante las validaciones se identificaron algunos registros inconsistentes causados por caracteres especiales y comillas presentes en el archivo CSV original. Este tipo de situaciones es común en procesos reales de ingestión de datos y demuestra la importancia de realizar validaciones posteriores a la carga.

# 5. Comparación entre SQL y Spark

| SQL | Spark (PySpark) |
|---|---|
| Más fácil de leer y escribir para consultas simples | Más flexible para procesamiento avanzado |
| Sintaxis declarativa y sencilla | Permite programación distribuida |
| Excelente integración con herramientas BI | Escalable para grandes volúmenes de datos |
| Ideal para consultas rápidas y agregaciones | Permite uso de APIs, UDFs y MLlib |
| Menor curva de aprendizaje | Requiere mayor conocimiento técnico |

## Ventajas de SQL

- Fácil interpretación de consultas.
- Excelente para análisis descriptivos.
- Muy utilizado en herramientas empresariales.
- Consultas rápidas y expresivas.

## Desventajas de SQL

- Menor flexibilidad para pipelines complejos.
- Limitaciones para procesamiento avanzado.
- Menor capacidad de personalización.

## Ventajas de Spark (PySpark)

- Procesamiento distribuido y escalable.
- APIs avanzadas para transformación de datos.
- Integración con Machine Learning mediante MLlib.
- Mayor control sobre procesos ETL.

## Desventajas de Spark (PySpark)

- Curva de aprendizaje más alta.
- Configuración más compleja.
- Puede requerir optimización de rendimiento.

# 6. Conclusiones

El desarrollo de la actividad permitió implementar un flujo completo de procesamiento de datos en Databricks utilizando Spark y SQL.

Se realizó la carga de un dataset desde Kaggle, el diseño de un esquema estructurado, la creación de tablas persistentes y diferentes validaciones analíticas utilizando tanto SQL como PySpark.

La práctica permitió comprender las ventajas del procesamiento distribuido con Spark y la facilidad de uso que ofrece SQL para consultas analíticas.